# Demo ELK + PostgreSQL

Ce notebook montre un flux simple:

Charger des produits dans PostgreSQL. 

Indexer ces produits dans Elasticsearch. 

Faire des recherches et agregations (Kibana peut ensuite les visualiser)

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "elasticsearch>=8,<9",
    "psycopg[binary]>=3.1",
    "pandas"
], check=True)
print("Dependencies installed")


In [ ]:
import os
from datetime import date

import pandas as pd
import psycopg
from psycopg.rows import dict_row
from elasticsearch import Elasticsearch, helpers

ES_URL = os.getenv("COURSE_ES_URL", "http://elasticsearch:9200")
PG_HOST = os.getenv("COURSE_PG_HOST", "postgres")
PG_DB = os.getenv("COURSE_PG_DB", "elk_course")
PG_USER = os.getenv("COURSE_PG_USER", "elk")
PG_PASSWORD = os.getenv("COURSE_PG_PASSWORD", "elkpass")
PG_PORT = os.getenv("COURSE_PG_PORT", "5432")

conninfo = f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASSWORD}"
print("ES_URL:", ES_URL)
print("PG:", f"{PG_USER}@{PG_HOST}:{PG_PORT}/{PG_DB}")


In [ ]:
products = [
    ("Casque audio Bluetooth", "Acme", 99.90, True, date(2026, 1, 10)),
    ("Clavier mecanique", "KeyPro", 129.00, False, date(2026, 1, 15)),
    ("Souris ergonomique", "Acme", 59.50, True, date(2026, 1, 20)),
    ("Ecran 27 pouces", "ViewMax", 249.99, True, date(2026, 2, 3)),
    ("Dock USB-C", "KeyPro", 79.00, True, date(2026, 2, 12)),
]

with psycopg.connect(conninfo, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS products (
                id SERIAL PRIMARY KEY,
                name TEXT NOT NULL,
                brand TEXT NOT NULL,
                price DOUBLE PRECISION NOT NULL,
                in_stock BOOLEAN NOT NULL,
                created_at DATE NOT NULL
            );
        """)
        cur.execute("TRUNCATE TABLE products RESTART IDENTITY;")
        cur.executemany(
            """
            INSERT INTO products (name, brand, price, in_stock, created_at)
            VALUES (%s, %s, %s, %s, %s)
            """,
            products,
        )

print(f"Inserted {len(products)} rows in PostgreSQL")


In [ ]:
with psycopg.connect(conninfo, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT id, name, brand, price, in_stock, created_at
            FROM products
            ORDER BY id
            """
        )
        rows = cur.fetchall()

df = pd.DataFrame(rows)
df


In [ ]:
es = Elasticsearch(ES_URL, request_timeout=30)
print("Elasticsearch version:", es.info()["version"]["number"])

index_name = "products_demo"
mapping = {
    "properties": {
        "name": {"type": "text"},
        "brand": {"type": "keyword"},
        "price": {"type": "float"},
        "in_stock": {"type": "boolean"},
        "created_at": {"type": "date"}
    }
}

if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)

es.indices.create(index=index_name, mappings=mapping)

actions = []
for row in rows:
    source = dict(row)
    source["created_at"] = source["created_at"].isoformat()
    actions.append({
        "_index": index_name,
        "_id": source["id"],
        "_source": source
    })

helpers.bulk(es, actions)
es.indices.refresh(index=index_name)
print(f"Indexed {len(actions)} documents in '{index_name}'")


In [ ]:
search_query = {
    "query": {
        "bool": {
            "must": [{"match": {"name": "clavier dock"}}],
            "filter": [{"term": {"in_stock": True}}]
        }
    }
}

search_res = es.search(index=index_name, body=search_query)
print("hits:", search_res["hits"]["total"])
for hit in search_res["hits"]["hits"]:
    print(hit["_id"], hit["_source"])


In [ ]:
agg_query = {
    "size": 0,
    "aggs": {
        "brands": {"terms": {"field": "brand"}},
        "avg_price": {"avg": {"field": "price"}}
    }
}

agg_res = es.search(index=index_name, body=agg_query)
agg_res["aggregations"]


## Kibana

Dans Kibana (`http://localhost:5601`), cree un Data View sur `products_demo` puis explore:
- Discover pour les documents
- Lens pour des graphes par marque et prix moyen
